# Prerequisite: Set Up Your SQL Environment

Run the cell below once before the lab, and again any time the notebook kernel restarts.

This notebook runs on a local Python kernel, which cannot run SQL by itself. The cell below sets that up:

1. **Installs the Snowflake connector** (`snowflake-connector-python`) into the notebook's Python environment.
2. **Connects to your lab account** by name. Your default connection may point to a different account, so the cell names the lab account explicitly.
3. **Sets the lab context**:
   - **Role `ACCOUNTADMIN`**: owns `SEAT_SWEEPER_LAB` and can create the dynamic tables, views and app objects. Your session otherwise starts as `ATTENDEE_ROLE`, which cannot.
   - **Warehouse `SEAT_SWEEPER_WH`**: the lab's X-Small warehouse, which auto-suspends after 60 seconds idle.
   - **Database `SEAT_SWEEPER_LAB`, schema `APP`**: where the pipeline and app objects live.
4. **Adds a `%%sql` helper** so the SQL cells in the lab run on Snowflake.

The output table shows your active context. Confirm it reads `ACCOUNTADMIN`, `SEAT_SWEEPER_WH`, `SEAT_SWEEPER_LAB` and `APP` before you continue to `lab_instructions.ipynb`.

> **Note:** This changes only your session. It creates and grants nothing in Snowflake.

### How to write SQL cells in this lab

Start every SQL cell with `%%sql` on its own first line. Everything below it runs on Snowflake. You can put several statements in one cell; the last result is shown.

**Comments:** use `--` for a single line or `/* ... */` for a block, placed **after** the `%%sql` line. Never use `#`, which is not valid in Snowflake SQL.

```sql
%%sql
-- This is a single-line SQL comment
/* This is a
   block comment */
SELECT CURRENT_ROLE();
```

> **If you see `SyntaxError: invalid syntax` pointing at a `--` line**, the comment is fine. The cell is missing `%%sql` on its first line, so Python tried to run it. Add `%%sql` and re-run it.
>
> **If you see `UsageError: Cell magic %%sql not found`**, the kernel restarted. Re-run this prerequisite cell.

In [ ]:
%pip install -q snowflake-connector-python pandas

import snowflake.connector
from IPython import get_ipython
from IPython.display import display
import pandas as pd
pd.set_option("display.max_colwidth", None)  # show full text (log messages, queries) instead of "..."

# The lab account's connection in ~/.snowflake/connections.toml.
# Your default connection may point to a different account, so name it explicitly.
CONNECTION = "REPLACE_WITH_YOUR_NAMED_CONNECTION"  # from: snow connection list

conn = snowflake.connector.connect(connection_name=CONNECTION)
for stmt in ("USE ROLE ACCOUNTADMIN", "USE WAREHOUSE SEAT_SWEEPER_WH",
             "USE DATABASE SEAT_SWEEPER_LAB", "USE SCHEMA APP"):
    conn.cursor().execute(stmt)

def sql(line, cell):
    """Run every statement in the cell on the lab connection; show the last result."""
    cur = None
    for cur in conn.execute_string(cell, remove_comments=True):
        pass
    if cur is not None and cur.description:
        display(pd.DataFrame(cur.fetchall(), columns=[c[0] for c in cur.description]))

get_ipython().register_magic_function(sql, magic_kind="cell", magic_name="sql")

display(pd.DataFrame(conn.cursor().execute(
    "SELECT CURRENT_ACCOUNT_NAME() AS account, CURRENT_USER() AS user_name, CURRENT_ROLE() AS role, "
    "CURRENT_WAREHOUSE() AS warehouse, CURRENT_DATABASE() AS database, CURRENT_SCHEMA() AS schema"
).fetchall(), columns=["ACCOUNT", "USER_NAME", "ROLE", "WAREHOUSE", "DATABASE", "SCHEMA"]))